# Exp 3-v10 🏁 — v8 Best Config + **Epoch 100** (마지막 시도)

## 전략
- v8 sweep best config 그대로 사용
- epoch 70 → 100으로 늘려서 Dev 0.7005가 Test에도 반영되도록
- 스케일 정규화 없음 (v9에서 역효과 확인)

## v8 Best Config
- hidden=1000 | lr=3.65e-05 | dropout=0.4 | batch=256 | epochs=**100** | wd=1e-4

## 근거
- v8에서 epoch 70 → Dev 0.7005 (70% 돌파)
- Test 69.13%로 gap 0.92%p 존재
- lr=3.65e-05 (매우 낮음) → epoch 더 길수록 천천히 수렴하는 경향
- epoch 100까지 학습하면 더 안정적인 수렴 기대

In [ ]:
!pip install datasets scikit-learn -q

In [ ]:
import torch, torch.nn as nn, torch.optim as optim, torch.backends.cudnn as cudnn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score
from datasets import load_dataset
from scipy.sparse import hstack, csr_matrix
import numpy as np, copy, re
SEED=42
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
cudnn.benchmark=False; cudnn.deterministic=True
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device:{device}')

In [ ]:
data=load_dataset('Sp1786/multiclass-sentiment-analysis-dataset')
def remove_empty(row):
    return all(row[f] not in [None,''] for f in ['id','text','label','sentiment'])
train_data=data['train'].filter(remove_empty)
dev_data=data['validation'].filter(remove_empty)
test_data=data['test'].filter(remove_empty)
output_size=len(set(train_data['label']))
train_labels=train_data['label']
test_labels_list=test_data['label']
print(f'Train:{len(train_data)}|Dev:{len(dev_data)}|Test:{len(test_data)}|Classes:{output_size}')

In [ ]:
def preprocess_text(text):
    text=text.lower()
    text=re.sub(r'http\S+|www\S+','',text)
    text=text.replace('`',"'")
    text=text.replace('****',' bad ').replace('***',' bad ')
    text=re.sub(r'!{3,}',' verymuch ! ',text)
    text=re.sub(r'(.)\1{3,}',r'\1\1',text)
    text=re.sub(r"won't",'will not',text)
    text=re.sub(r"can't",'cannot',text)
    text=re.sub(r"n't",' not',text)
    text=re.sub(r"'re",' are',text)
    text=re.sub(r"'ve",' have',text)
    text=re.sub(r"'ll",' will',text)
    text=re.sub(r"'d",' would',text)
    text=re.sub(r"'m",' am',text)
    for pat,rep in [
        (r'\bidk\b','i do not know'),(r'\bur\b','your'),
        (r'\bnaw\b','no'),(r'\bgonna\b','going to'),
        (r'\bwanna\b','want to'),(r'\blol\b','laughing'),
        (r'\bomg\b','oh my god'),(r'\bwtf\b','what the'),
        (r'\bugh\b','disgusting'),(r'\btho\b','though'),
        (r'\bkinda\b','kind of'),(r'\bcuz\b','because'),
        (r'\bsoo+\b','so'),(r'\bthx\b','thanks'),
        (r'\byep\b','yes'),(r'\byup\b','yes'),
        (r'\bnope\b','no'),(r'\btbh\b','to be honest'),
        (r'\bimo\b','in my opinion'),
    ]:
        text=re.sub(pat,rep,text)
    return text

def extract_handcraft(texts):
    features=[]
    for text in texts:
        t=str(text); tl=t.lower(); words=t.split()
        features.append([
            min(t.count('!'),5),
            min(t.count('?'),5),
            sum(1 for w in words if w.isupper() and len(w)>1),
            min(len(words),50),
            int(bool(re.search(r'http\S+',tl))),
            int(any(e in tl for e in [':)',':(',':d',':/','haha','hehe','lmao'])),
        ])
    return np.array(features,dtype=np.float32)

vectorizer=TfidfVectorizer(max_features=30000,preprocessor=preprocess_text,min_df=2)
vectorizer.fit(train_data['text'])

def build_features(data_split):
    tfidf_mat=vectorizer.transform(data_split['text'])
    hc_mat=csr_matrix(extract_handcraft(data_split['text']))
    return torch.FloatTensor(hstack([tfidf_mat,hc_mat]).toarray()).to(device)

train_t=build_features(train_data)
dev_t=build_features(dev_data)
test_t=build_features(test_data)
dev_labels_t=torch.tensor(dev_data['label'],dtype=torch.long).to(device)
input_size=train_t.shape[1]
print(f'입력 크기:{input_size}')

In [ ]:
class MLP(nn.Module):
    def __init__(self,i,h,o,d=0.0):
        super().__init__()
        self.fc1=nn.Linear(i,h)
        self.fc2=nn.Linear(h,h//2)
        self.fc3=nn.Linear(h//2,o)
        self.activation=nn.GELU()
        self.output_act=nn.Softmax(dim=1)
        self.dropout=nn.Dropout(p=d)
    def forward(self,x):
        x=self.dropout(self.activation(self.fc1(x)))
        x=self.dropout(self.activation(self.fc2(x)))
        return self.output_act(self.fc3(x))

In [ ]:
# v8 best config + epoch 100
H,LR,D,WD,EP,BS = 1000, 3.649238347809796e-05, 0.4, 1e-4, 100, 256
print(f'Config: hidden={H} | lr={LR:.2e} | dropout={D} | wd={WD} | epochs={EP}')

torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
model=MLP(input_size,H,output_size,D).to(device)
opt=optim.Adam(model.parameters(),lr=LR,weight_decay=WD)
lfn=nn.CrossEntropyLoss()
best_dev,best_state=0,None
print('🚀 학습 시작...')
print('='*50)
for epoch in range(EP):
    model.train()
    for i in range(0,len(train_t),BS):
        bd=train_t[i:i+BS]
        bl=torch.tensor(train_labels[i:i+BS],device=device)
        loss=lfn(model(bd),bl)
        opt.zero_grad(); loss.backward(); opt.step()
    model.eval()
    with torch.no_grad():
        da=(torch.argmax(model(dev_t),dim=1)==dev_labels_t).float().mean().item()
    if da>best_dev:
        best_dev,best_state=da,copy.deepcopy(model.state_dict())
        print(f'✨ Epoch {epoch+1}/{EP}|Dev:{da:.4f}|NEW BEST!')
    elif (epoch+1)%10==0:
        print(f'Epoch {epoch+1}/{EP}|Dev:{da:.4f}')
print('='*50)
model.load_state_dict(best_state)
torch.save(best_state,'best_model_exp3_v10_final.pt')
with torch.no_grad():
    test_acc=accuracy_score(test_labels_list,torch.argmax(model(test_t),dim=1).cpu().tolist())
print(f'\n✅ 저장: best_model_exp3_v10_final.pt')
print(f'📊 Dev:{best_dev:.4f}|Test:{test_acc*100:.2f}%')
print(f'\n🎯 목표 70%: {"달성! 🎉🎉🎉" if test_acc>=0.70 else f"{test_acc*100:.2f}% (v8 69.13% 대비 {(test_acc-0.6913)*100:+.2f}%p)"}')

In [ ]:
from google.colab import files
files.download('best_model_exp3_v10_final.pt')